# OUTCAR → DeePMD converter (Colab)

Converts VASP `OUTCAR` files into DeePMD-kit `npy` training data — every ionic step
becomes a frame, real periodic boxes, **no unit conversion** (VASP is already eV/eV·Å⁻¹/Å).

**Flow:** (1) get your OUTCARs into Colab (upload *or* Drive), (2) set parameters,
(3) run, (4) download/save the result (Colab is ephemeral — export or it's lost).

Pure Python + NumPy, nothing to install.

## 1 · Get your OUTCAR files into Colab
Pick **one** of the two cells below.

## Stage 1: Google Drive Mount

In [5]:
from google.colab import drive
drive.mount('/content/drive')
# Point to folder where the OUTCARS reside
OUTCAR_SOURCE = '/content/drive/MyDrive/vasp'
import glob, os
found = glob.glob(os.path.join(OUTCAR_SOURCE, '**', 'OUTCAR*'), recursive=True)
print(f'found {len(found)} OUTCAR files under {OUTCAR_SOURCE}')
for f in found[:10]: print('  ', f)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
found 1 OUTCAR files under /content/drive/MyDrive/vasp
   /content/drive/MyDrive/vasp/OUTCAR.1


## Stage 2: Conversion


In [6]:
%%writefile outcar2deepmd.py
#!/usr/bin/env python3
"""
outcar2deepmd.py - convert VASP OUTCAR files into DeePMD-kit (.npy) training data.

VASP is already in DeePMD's native units (eV, eV/A, A), hence no unit conversion is required.
Every snapshot in the AIMD trajectory becomes a training frame for DeePMD, grouped by compostion.

Usage:
    python outcar2deepmd.py OUTCAR [OUTCAR2 ...] \
        --type_map Cu In C H --out deepmd_data \
        [--skip 50] [--stride 5] [--virial]

  --type_map : GLOBAL, canonical element order shared across ALL your data.
               type.raw indices are assigned from this, so mixing OUTCARs with
               different species subsets stays consistent (essential for training
               one model on multiple systems).
  --skip     : drop the first N ionic steps per OUTCAR (AIMD equilibration burn-in).
  --stride   : keep every Nth step (decorrelate highly-correlated AIMD frames).
  --virial   : also extract the stress -> virial (useful for bulk/surface, Cu-In).
"""
import argparse, os, re, glob
import numpy as np


# OUTCAR parsing
def parse_species(text):
    """Element symbols in POTCAR order + count per species."""
    # elements from 'VRHFIN =Cu: ...'  (clean, one per species, in order)
    els = re.findall(r"VRHFIN\s*=\s*([A-Za-z]+)\s*:", text)
    if not els:                                   # fallback: POTCAR title lines
        raw = re.findall(r"POTCAR:\s+\S+\s+(\S+)", text)
        seen, els = set(), []
        for r in raw:
            e = r.split("_")[0]
            if e not in seen:
                seen.add(e); els.append(e)
    m = re.search(r"ions per type\s*=\s*([\d\s]+)", text)
    counts = [int(x) for x in m.group(1).split()] if m else []
    return els, counts


def parse_frames(text):
    """Yield (cell 3x3, positions Nx3, forces Nx3, energy) for each ionic step."""
    lines = text.splitlines()
    cell = None
    i = 0
    pending_cell = None
    while i < len(lines):
        ln = lines[i]
        # lattice vectors update (appears each ionic step; may change in NPT)
        if "direct lattice vectors" in ln:
            try:
                pending_cell = np.array(
                    [[float(x) for x in lines[i + k + 1].split()[:3]] for k in range(3)]
                )
            except (ValueError, IndexError):
                pass
            i += 4
            continue
        # position/force block
        if "POSITION" in ln and "TOTAL-FORCE" in ln:
            j = i + 2                              # skip the dashed separator
            pos, frc = [], []
            while j < len(lines) and "---" not in lines[j]:
                p = lines[j].split()
                if len(p) >= 6:
                    pos.append([float(p[0]), float(p[1]), float(p[2])])
                    frc.append([float(p[3]), float(p[4]), float(p[5])])
                j += 1
            # energy: next 'energy(sigma->0)' after this block
            energy = None
            k = j
            while k < len(lines):
                if "energy(sigma->0)" in lines[k]:
                    energy = float(lines[k].split("energy(sigma->0)")[1].split("=")[1].split()[0])
                    break
                k += 1
            if pos and energy is not None:
                c = pending_cell if pending_cell is not None else cell
                cell = c
                yield (np.array(c), np.array(pos), np.array(frc), energy)
            i = j
            continue
        i += 1


def parse_stress(text):
    """Yield the stress tensor (kBar, Voigt xx yy zz xy yz zx) per ionic step, if present."""
    out = []
    for ln in text.splitlines():
        if "in kB" in ln:
            v = ln.split("in kB")[1].split()
            if len(v) >= 6:
                out.append([float(x) for x in v[:6]])
    return out


# Conversion
def outcar_to_frames(path, type_map, skip=0, stride=1, want_virial=False):
    text = open(path, errors="ignore").read()
    els, counts = parse_species(text)
    if not els or not counts:
        raise RuntimeError(f"{path}: could not read species/counts")
    # Per-atom global type indices (from the canonical type_map)
    atom_types = []
    for e, n in zip(els, counts):
        if e not in type_map:
            raise RuntimeError(f"{path}: element {e} not in --type_map {type_map}")
        atom_types += [type_map.index(e)] * n
    atom_types = np.array(atom_types)
    natoms = len(atom_types)

    frames = list(parse_frames(text))
    frames = frames[skip::stride]
    stresses = parse_stress(text)[skip::stride] if want_virial else None

    rows = []
    for idx, (cell, pos, frc, en) in enumerate(frames):
        if len(pos) != natoms:
            continue
        vir = None
        if want_virial and stresses and idx < len(stresses):
            # stress(kBar)->eV/A^3: 1 kBar = 0.1 GPa = 6.2415e-4 eV/A^3 ; virial = -stress*V
            s = np.array(stresses[idx]) * 6.241509074e-4
            S = np.array([[s[0], s[3], s[5]], [s[3], s[1], s[4]], [s[5], s[4], s[2]]])
            V = abs(np.linalg.det(cell))
            vir = (-S * V).reshape(-1)
        rows.append((cell.reshape(-1), pos.reshape(-1), frc.reshape(-1), en, vir))
    # composition signature -> groups frames from different OUTCARs into same system
    sig = tuple(atom_types.tolist())
    return sig, atom_types, rows


def write_system(outdir, atom_types, rows, type_map, want_virial):
    s = os.path.join(outdir, "set.000")
    os.makedirs(s, exist_ok=True)
    box = np.array([r[0] for r in rows])
    coord = np.array([r[1] for r in rows])
    force = np.array([r[2] for r in rows])
    energy = np.array([r[3] for r in rows])
    np.save(os.path.join(s, "box.npy"), box)
    np.save(os.path.join(s, "coord.npy"), coord)
    np.save(os.path.join(s, "force.npy"), force)
    np.save(os.path.join(s, "energy.npy"), energy)
    if want_virial and all(r[4] is not None for r in rows):
        np.save(os.path.join(s, "virial.npy"), np.array([r[4] for r in rows]))
    open(os.path.join(outdir, "type.raw"), "w").write("\n".join(map(str, atom_types)) + "\n")
    open(os.path.join(outdir, "type_map.raw"), "w").write("\n".join(type_map) + "\n")
    return len(rows)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("outcars", nargs="+")
    ap.add_argument("--type_map", nargs="+", required=True)
    ap.add_argument("--out", default="deepmd_data")
    ap.add_argument("--skip", type=int, default=0)
    ap.add_argument("--stride", type=int, default=1)
    ap.add_argument("--virial", action="store_true")
    a = ap.parse_args()

    # expand globs/dirs
    paths = []
    for p in a.outcars:
        if os.path.isdir(p):
            paths += glob.glob(os.path.join(p, "**", "OUTCAR*"), recursive=True)
        else:
            paths += glob.glob(p)
    paths = sorted(set(paths))
    if not paths:
        raise SystemExit("no OUTCAR files found")

    groups = {}   # signature -> (atom_types, [rows])
    for p in paths:
        sig, atypes, rows = outcar_to_frames(p, a.type_map, a.skip, a.stride, a.virial)
        if not rows:
            print(f"  {p}: 0 frames (check skip/stride)"); continue
        if sig not in groups:
            groups[sig] = (atypes, [])
        groups[sig][1].extend(rows)
        print(f"  {p}: {len(rows)} frames  ({len(atypes)} atoms)")

    os.makedirs(a.out, exist_ok=True)
    total = 0
    for k, (sig, (atypes, rows)) in enumerate(groups.items()):
        d = os.path.join(a.out, f"sys_{k:03d}")
        n = write_system(d, atypes, rows, a.type_map, a.virial)
        comp = "".join(f"{a.type_map[t]}{atypes.tolist().count(t)}" for t in sorted(set(atypes)))
        print(f"system {d}: {n} frames, composition {comp}")
        total += n
    print(f"\nDONE: {total} frames across {len(groups)} system(s) -> {a.out}/")
    print(f"train with  \"systems\": [{', '.join(repr(os.path.join(a.out, f'sys_{k:03d}')) for k in range(len(groups)))}]")


if __name__ == "__main__":
    main()


Writing outcar2deepmd.py


In [7]:
# import the functions so we can call them directly in the notebook
import importlib, outcar2deepmd as o2d
importlib.reload(o2d)
print('converter loaded')

converter loaded


## 3 · Set parameters and convert
- **TYPE_MAP** — GLOBAL canonical element order, shared across *all* your data. Set once, never reorder.
- **SKIP / STRIDE** — drop AIMD burn-in / decorrelate correlated frames.
- **VIRIAL** — also extract stress→virial (bulk/surface).

In [10]:
import glob, os

TYPE_MAP = ['Cu','In']                      # element order
OUT      = 'deepmd_data'
SKIP     = 0
STRIDE   = 1
VIRIAL   = False

if os.path.isdir('outcars') and glob.glob('outcars/**/OUTCAR*', recursive=True):
    PATHS = glob.glob('outcars/**/OUTCAR*', recursive=True)
else:
    PATHS = glob.glob(os.path.join(OUTCAR_SOURCE, '**', 'OUTCAR*'), recursive=True)  # from Option B
PATHS = sorted(set(PATHS))
print(f'{len(PATHS)} OUTCAR(s) to convert')

groups = {}
for p in PATHS:
    sig, atypes, rows = o2d.outcar_to_frames(p, TYPE_MAP, SKIP, STRIDE, VIRIAL)
    if not rows: print(f'  {p}: 0 frames'); continue
    groups.setdefault(sig, (atypes, []))[1].extend(rows)
    print(f'  {os.path.basename(os.path.dirname(p)) or p}: {len(rows)} frames, {len(atypes)} atoms')

os.makedirs(OUT, exist_ok=True)
total = 0
for k, (sig, (atypes, rows)) in enumerate(groups.items()):
    d = os.path.join(OUT, f'sys_{k:03d}')
    n = o2d.write_system(d, atypes, rows, TYPE_MAP, VIRIAL)
    comp = ''.join(f'{TYPE_MAP[t]}{atypes.tolist().count(t)}' for t in sorted(set(atypes)))
    print(f'system {d}: {n} frames, {comp}'); total += n
print(f'\nDONE: {total} frames, {len(groups)} system(s) -> {OUT}/')
print('train with  "systems":', [os.path.join(OUT, f'sys_{k:03d}') for k in range(len(groups))])

1 OUTCAR(s) to convert
  vasp: 4935 frames, 60 atoms
system deepmd_data/sys_000: 4935 frames, Cu30In30

DONE: 4935 frames, 1 system(s) -> deepmd_data/
train with  "systems": ['deepmd_data/sys_000']


## 4 · Download / save the result
Colab is ephemeral — export or it's gone.

In [ ]:
# zip the dataset and download it
import shutil
shutil.make_archive('deepmd_data', 'zip', 'deepmd_data')
try:
    from google.colab import files
    files.download('deepmd_data.zip')
except Exception as e:
    print('download skipped (not in Colab?):', e)
# or copy to Drive (if mounted):
# shutil.copytree('deepmd_data', '/content/drive/MyDrive/deepmd_data', dirs_exist_ok=True)

## 5 · (optional) Cross-check against dpdata
Validate this converter against the community-standard tool on one OUTCAR the first time.

In [ ]:
# pip install dpdata, then compare energies/forces on the first OUTCAR
!pip -q install dpdata
import dpdata, numpy as np, glob
p = PATHS[0]
d = dpdata.LabeledSystem(p, fmt='vasp/outcar')
d.to_deepmd_npy('dpdata_check')
e_mine = np.load(sorted(glob.glob('deepmd_data/sys_000/set.000/energy.npy'))[0])
e_dp   = np.load(sorted(glob.glob('dpdata_check/**/energy.npy', recursive=True))[0])
print('my energies[:3] :', e_mine[:3])
print('dpdata energies :', e_dp[:3])
print('max |diff| (should be ~0):', float(np.abs(np.sort(e_mine)[:len(e_dp)] - np.sort(e_dp)).max()))